# Chapter 5

In [ ]:
# Input question about text --> 
# ---> Retrieval the part of the text with more fit with our question -->
# ---> Create a better prompt join this part of tet as context.

In [2]:
# Text --> Chunks --> Stored with FAISS

### Embeddings

In [7]:
from openai import OpenAI
client = OpenAI()

# Function to get the vector embedding for a given text
def get_vector_embeddings(text):
    response = client.embeddings.create(
        input=text,
        model="text-embedding-ada-002"
    )
    embeddings = [r.embedding for r in response.data]
    return embeddings[0]

### Chunk correctly

In [6]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=100, # 100 tokens
    chunk_overlap=20, # 20 tokens of overlap
)

text = """
Welcome to the "Unicorn Enterprises: Where Magic Happens"
Employee Handbook! We're thrilled to have you join our team
of dreamers, doers, and unicorn enthusiasts. At Unicorn
Enterprises, we believe that work should be as enchanting as
it is productive. This handbook is your ticket to the
magical world of our company, where we'll outline the
principles, policies, and practices that guide us on this
extraordinary journey. So, fasten your seatbelts and get
ready to embark on an adventure like no other!
...
As we conclude this handbook, remember that at Unicorn
Enterprises, the pursuit of excellence is a never-ending
quest. Our company's success depends on your passion,
creativity, and commitment to making the impossible
possible. We encourage you to always embrace the magic
within and outside of work, and to share your ideas and
innovations to keep our enchanted journey going. Thank you
for being a part of our mystical family, and together, we'll
continue to create a world where magic and business thrive
hand in hand!
"""

chunks = text_splitter.split_text(text=text)
print(chunks[0])

Welcome to the "Unicorn Enterprises: Where Magic Happens"
Employee Handbook! We're thrilled to have you join our team
of dreamers, doers, and unicorn enthusiasts. At Unicorn
Enterprises, we believe that work should be as enchanting as
it is productive. This handbook is your ticket to the
magical world of our company, where we'll outline the
principles, policies, and practices that guide us on this


### FAISS 

In [4]:
import numpy as np
import faiss

In [14]:
# The get_vector_embeddings function is defined in a preceding example
emb = [get_vector_embeddings(chunk) for chunk in chunks]
vectors = np.array(emb)

# Create a FAISS index
index = faiss.IndexFlatL2(vectors.shape[1])
index.add(vectors)

# Function to perform a vector search
def vector_search(query_text, k=2):
    query_vector = get_vector_embeddings(query_text)
    distances, indices = index.search(
        np.array([query_vector]), k)
    # print(f'Array Distancias: {distances}')
    # print(f'Array indices: {indices}')
    return [(chunks[i], float(dist)) for dist,
        i in zip(distances[0], indices[0])]
    
# Example search
user_query = "do you suggest to invest in Unicorn's company?"
search_results = vector_search(user_query)
print(f"Search results for {user_query}:", search_results)

Search results for do you suggest to invest in Unicorn's company?: [('Welcome to the "Unicorn Enterprises: Where Magic Happens"\nEmployee Handbook! We\'re thrilled to have you join our team\nof dreamers, doers, and unicorn enthusiasts. At Unicorn\nEnterprises, we believe that work should be as enchanting as\nit is productive. This handbook is your ticket to the\nmagical world of our company, where we\'ll outline the\nprinciples, policies, and practices that guide us on this', 0.3820256292819977), ("principles, policies, and practices that guide us on this\nextraordinary journey. So, fasten your seatbelts and get\nready to embark on an adventure like no other!\n...\nAs we conclude this handbook, remember that at Unicorn\nEnterprises, the pursuit of excellence is a never-ending\nquest. Our company's success depends on your passion,\ncreativity, and commitment to making the impossible", 0.3852657675743103)]


### Join all 

In [13]:
# Insert in the prompt

# Function to perform a vector search and then ask # GPT-3.5-turbo a question
def search_and_chat(user_query, k=1):
    # Perform the vector search
    search_results = vector_search(user_query, k)
    #print(f"Search results: {search_results}\n\n")
    
    prompt_with_context = f"""Context:{search_results}\
    Answer the question: {user_query}"""
    
    # Create a list of messages for the chat
    messages = [
        {"role": "system", "content": """Please answer the
        questions provided by the user. Use only the context
        provided to you to respond to the user, if you don't
        know the answer say \"I don't know\"."""},
        {"role": "user", "content": prompt_with_context},
    ]
        
    # Get the model's response
    response = client.chat.completions.create(
        model="gpt-3.5-turbo", messages=messages)
    
    # Print the assistant's reply
    print(f"""Response:
    {response.choices[0].message.content}""")

# Example search and chat
search_and_chat("do you suggest to invest in Unicorn' company?")

Array Distancias: [[0.37612656]]
Array indices: [[0]]
Response:
    Based on the context provided in the employee handbook of "Unicorn Enterprises: Where Magic Happens," it seems like a lively and enthusiastic company that values a magical work environment. However, as an AI assistant, I cannot make investment suggestions. I recommend conducting thorough research, including financial assessments and industry analysis, before making any investment decisions.


In [2]:
from langchain_community.vectorstores.faiss import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

## Lang Chain

In [3]:
# 1. Create the documents:
documents = [
    "James Phoenix worked at JustUnderstandingData.",
    "James Phoenix currently is 31 years old.",
    """Data engineering is the designing and building systems for collecting,
    storing, and analyzing data at scale."""
]

# 2. Create a vectorstore:
vectorstore = FAISS.from_texts(texts=documents, embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever()

# 3. Create a prompt:
template = """Answer the question based only on the following context:
---
Context: {context}
---
Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

# 4. Create a chat model:
model = ChatOpenAI()

In [ ]:
# 5. Create the chain
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

# 6. Call chain
chain.invoke("What is data engineering?")
# 'Data engineering is the process of designing and building systems for
# collecting, storing, and analyzing data at scale.'

chain.invoke("Who is James Phoenix?")
# 'Based on the given context, James Phoenix is a 31-year-old individual who
# worked at JustUnderstandingData.'

chain.invoke("What is the president of the US?")
# I don't know


## Pinnecone

In [4]:
from pinecone import Pinecone, ServerlessSpec

In [13]:
index_name = "employee-handbook"
environment = "us-east-1"
pc = Pinecone() # This reads the PINECONE_API_KEY env var

In [14]:
# Check if index already exists:
# (it shouldn't if this is first time)
if index_name not in pc.list_indexes().names():
    # if does not exist, create index
    pc.create_index(
        index_name,
        # Using the same vector dimensions as text-embedding-ada-002
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region=environment),
    )
    print("Creado")
# Connect to index:
index = pc.Index(index_name)

# View index stats:
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'namespaces': {},
 'total_vector_count': 0}

## Store yours vectors 

In [17]:
from tqdm import tqdm # For printing a progress bar
from time import sleep

# How many embeddings you create and insert at once
batch_size = 1
retry_limit = 5 # maximum number of retries

for i in tqdm(range(0, len(chunks), batch_size)):
    # Find end of batch
    i_end = min(len(chunks), i+batch_size)
    meta_batch = chunks[i:i_end]
    # Get ids
    ids_batch = [str(j) for j in range(i, i_end)]
    # Get texts to encode
    texts = [x for x in meta_batch]
    # Create embeddings
    # (try-except added to avoid RateLimitError)
    done = False
    try:
        # Retrieve embeddings for the whole batch at once
        embeds = []
        for text in texts:
            embedding = get_vector_embeddings(text)
            embeds.append(embedding)
        done = True
    except:
        retry_count = 0
        while not done and retry_count < retry_limit:
            try:
                for text in texts:
                    embedding = get_vector_embeddings(text)
                    embeds.append(embedding)
                done = True
            except:
                sleep(5)
                retry_count += 1
    if not done:
        print(f"""Failed to get embeddings after
        {retry_limit} retries.""")
        continue
        
    # Cleanup metadata
    meta_batch = [{
        'batch': i,
        'text': x
    } for x in meta_batch]
    to_upsert = list(zip(ids_batch, embeds, meta_batch))
    # Upsert to Pinecone
    index.upsert(vectors=to_upsert)

100%|████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:03<00:00,  1.10s/it]


In [18]:
# Retrieve from Pinecone
user_query = "do we get free unicorn rides?"
def pinecone_vector_search(user_query, k):
    xq = get_vector_embeddings(user_query)
    res = index.query(vector=xq, top_k=k, include_metadata=True)
    return res
    
pinecone_vector_search(user_query, k=1)

{'matches': [{'id': '1',
              'metadata': {'batch': 1.0,
                           'text': 'principles, policies, and practices that '
                                   'guide us on this\n'
                                   'extraordinary journey. So, fasten your '
                                   'seatbelts and get\n'
                                   'ready to embark on an adventure like no '
                                   'other!\n'
                                   '...\n'
                                   'As we conclude this handbook, remember '
                                   'that at Unicorn\n'
                                   'Enterprises, the pursuit of excellence is '
                                   'a never-ending\n'
                                   "quest. Our company's success depends on "
                                   'your passion,\n'
                                   'creativity, and commitment to making the '
                      

## Self Quering with Lark

In [23]:
from langchain_core.documents import Document
from langchain_community.vectorstores.chroma import Chroma
from langchain_openai import OpenAIEmbeddings
import lark
import getpass
import os
import warnings

# Disabling warnings:
# warnings.filterwarnings("ignore")

In [24]:
from langchain_openai.chat_models import ChatOpenAI
from langchain.retrievers.self_query.base \
import SelfQueryRetriever
from langchain.chains.query_constructor.base \
import AttributeInfo

In [25]:
docs = [
    Document(
        page_content="A tale about a young wizard and his \
            journey in a magical school.",
        metadata={
            "title": "Harry Potter and the Philosopher's Stone",
            "author": "J.K. Rowling",
            "year_published": 1997,
            "genre": "Fiction",
            "isbn": "978-0747532699",
            "publisher": "Bloomsbury",
            "language": "English",
            "page_count": 223,
            "summary": "The first book in the Harry Potter \
            series where Harry discovers his magical \
            heritage.",
            "rating": 4.8,
        },
    ),
    # ... More documents ...
]

In [ ]:
# Create the embeddings and vectorstore:
embeddings = OpenAIEmbeddings()
vectorstore = Chroma.from_documents(docs, OpenAIEmbeddings())

# Basic Info
basic_info = [
    AttributeInfo(name="title", description="The title of the book",
    type="string"),
    AttributeInfo(name="author", description="The author of the book",
    type="string"),
    AttributeInfo(
        name="year_published",
        description="The year the book was published",
        type="integer",
    ),
]

# Detailed Info
detailed_info = [
    AttributeInfo(
        name="genre", description="The genre of the book",
        type="string or list[string]"
    ),
    AttributeInfo(
        name="isbn",
        description="The International Standard Book Number for the book",
    type="string",
    ),
    AttributeInfo(
        name="publisher",
        description="The publishing house that published the book",
        type="string",
    ),
    AttributeInfo(
        name="language",
        description="The primary language the book is written in",
        type="string",
    ),
    AttributeInfo(
        name="page_count", description="Number of pages in the book",
        type="integer"
    ),
]

# Analysis
analysis = [
    AttributeInfo(
        name="summary",
        description="A brief summary or description of the book",
        type="string",
    ),
    AttributeInfo(
        name="rating",
        description="""An average rating for the book (from reviews), ranging
        from 1-5""",
        type="float",
    ),
]
# Combining all lists into metadata_field_info
metadata_field_info = basic_info + detailed_info + analysis

In [ ]:
document_content_description = "Brief summary of a movie"
llm = ChatOpenAI(temperature=0)
retriever = SelfQueryRetriever.from_llm(
    llm, vectorstore, document_content_description, metadata_field_info
)

# Looking for sci-fi books
retriever.invoke("What are some sci-fi books?")
# [Document(page_content='''A futuristic society where firemen burn books to
# maintain order.''', metadata={'author': 'Ray Bradbury', 'genre': '...
# More documents..., truncated for brevity